In [ ]:
import numpy as np
from numpy.random import randn
import math # возможно понадобится

class myRNN:

    def __init__(self, input_size, output_size, hidden_size):
        # Инициализируйте веса. Помните, что между весами должно происходить умножение.
        # Вспомните, какие условия накладываются на матрицы. которые умножаются между собой.

        self.Whh = np.random.randn(hidden_size, hidden_size) * 0.5
        self.Wxh = np.random.randn(hidden_size, input_size) * 0.5
        self.Why = np.random.randn(output_size, hidden_size) * 0.5

        #Смещение, равное нулю
        self.bh = np.zeros((hidden_size, 1))
        self.by = np.zeros((output_size, 1))

    def forward(self, inputs):
        h = np.zeros((self.Whh.shape[0], 1))
        self.last_hs = { 0: h }
        self.last_inputs = inputs # может понадобится вам в backprop, здесь необязательно

        # Применила формулы с картинки
        for i, x in enumerate(inputs):
            x = np.array(x).reshape(-1, 1)
            h = np.tanh(np.dot(self.Wxh, x) + np.dot(self.Whh, h) + self.bh)
            self.last_hs[i + 1] = h  #Сохраняем для следующего шага

            y = np.dot(self.Why, h) + self.by

        return y, h

    def backprop(self, d_y, learn_rate):
        h = self.last_hs[1]
        h_previous = self.last_hs[0]
        x = self.last_inputs[0]

        #Градиенты для выходного слоя
        d_Why = d_y @ h.T
        d_by = d_y
        grad_from_output = self.Why.T @ d_y

        # Производная гиперболического тангенса
        d_tanh = 1 - h ** 2

        d_h = grad_from_output * d_tanh

        #Посчитанные градиенты
        d_Wxh = d_h @ x.T
        d_Whh = d_h @ h_previous.T


        #Обновление весов с учетом learning rate
        self.Wxh -= learn_rate * d_Wxh
        self.Whh -= learn_rate * d_Whh





In [ ]:
#Проверка работоспособности класса на примере использования модели для предсказания следующего символа в тексте
text = "Alice was beginning to get very tired of sitting by her sister on the bank, and of having nothing to do: once or twice she had peeped into the book her sister was reading, but it had no pictures or conversations in it, ‘and what is the use of a book,’ thought Alice ‘without pictures or conversation?’"
chars = sorted(set(text))
vocab_size = len(chars)
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

#Создаем мешок слов
def one_hot(idx, size):
    vec = np.zeros((size, 1))
    vec[idx] = 1
    return vec

X = []  # список векторов
Y = []  # список индексов следующих символов

for i in range(len(text) - 1):
    # Получаем текущий символ и его индекс
    current_char = text[i]
    current_idx = char_to_idx[current_char]
    # Кодируем текущий символ
    one_hot_vector = one_hot(current_idx, vocab_size)
    X.append(one_hot_vector)
    #Получаем следующий индекс
    next_char = text[i + 1]
    next_idx = char_to_idx[next_char]
    Y.append(next_idx)


In [ ]:
#Вручную воспроизводим функцию softmax
def softmax(x):
    exp_x = np.exp(x - np.max(x))
    return exp_x / exp_x.sum()

# Параметры модели, делающие обучение стабильным и не занимают много времени при воспроизведении кода
hidden_size = 32
learning_rate = 0.05
epochs = 500

# Создаем модель
model = myRNN(vocab_size, vocab_size, hidden_size)

losses = []

# Цикл обучения по эпохам, на первом шаге задаю h_prev=0
for epoch in range(epochs):
    total_loss = 0
    h_prev = np.zeros((hidden_size, 1))

    # Цикл прохождения по символам
    for symbol in range(len(X)):
        x = [X[symbol]]
        y_true_idx = Y[symbol]
        y_pred, h = model.forward(x)

        # Вычисляем loss
        probs = softmax(y_pred)
        loss = -np.log(probs[y_true_idx][0] + 1e-6) # вычисляем loss и добавляем очень малое число для избежания нуля внутри логарифма
        total_loss += loss

        d_y = probs.copy()
        d_y[y_true_idx] -= 1  # производная cross-entropy по входу softmax (отнимает 1 от правильного варианта)


        model.backprop(d_y, learning_rate)
        h_prev = h

    # Средняя ошибка всех символов за эпоху
    avg_loss = total_loss / len(X)
    losses.append(avg_loss)


    if epoch % 50 == 0:
        print(f"Эпоха {epoch}, Средняя ошибка: {avg_loss:.2f}")


Эпоха 0, Средняя ошибка: 3.54
Эпоха 50, Средняя ошибка: 1.93
Эпоха 100, Средняя ошибка: 1.88
Эпоха 150, Средняя ошибка: 1.87
Эпоха 200, Средняя ошибка: 1.86
Эпоха 250, Средняя ошибка: 1.85
Эпоха 300, Средняя ошибка: 1.85
Эпоха 350, Средняя ошибка: 1.84
Эпоха 400, Средняя ошибка: 1.84
Эпоха 450, Средняя ошибка: 1.84
